# Test Evaluation

This notebook compares Random, GeneticBatch, and SAC_Pretrained on the eight 1,000-timestep test datasets.

Generate all three result summaries before running the notebook:

```bash
PYTHONPATH=src .venv/bin/python src/evaluation.py --algorithm all
```

In [ ]:
from pathlib import Path
import csv
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-vec-cache")

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "evaluation.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root()
RESULTS_ROOT = PROJECT_ROOT / "outputs" / "evaluation"
SUMMARY_FILES = {
    "Random": RESULTS_ROOT / "random" / "test_random_summary.csv",
    "GeneticBatch": RESULTS_ROOT / "genetic" / "test_genetic_summary.csv",
    "SAC_Pretrained": RESULTS_ROOT / "sac_pretrained" / "test_sac_pretrained_summary.csv",
}

print("Results root:", RESULTS_ROOT)

In [ ]:
def load_summary(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing evaluation summary: {path}")
    with path.open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


summaries = {name: load_summary(path) for name, path in SUMMARY_FILES.items()}

for algorithm, rows in summaries.items():
    print(f"\n{algorithm}")
    for row in rows:
        print(
            f"{row['scenario_group']:11s} tasks={int(row['total_tasks']):6d} "
            f"miss={int(row['deadline_misses']):5d} loss={int(row['packet_losses']):5d} "
            f"latency={float(row['avg_latency']):.4f} energy={float(row['avg_energy']):.4f}"
        )

## Required Metrics

In [ ]:
SCENARIOS = ["BASE", "RAIN", "SNOW", "FOG", "FAST_MIXED", "SLOW_MIXED", "RANDOM_MIX_1", "RANDOM_MIX_2"]
ALGORITHMS = ["Random", "GeneticBatch", "SAC_Pretrained"]
COLORS = ["#4f83cc", "#62a87c", "#b86b4b"]
summary_by_algorithm = {
    algorithm: {row["scenario_group"]: row for row in rows}
    for algorithm, rows in summaries.items()
}


def plot_metric(metric: str, title: str, ylabel: str):
    x = np.arange(len(SCENARIOS))
    width = 0.25
    fig, axis = plt.subplots(figsize=(15, 4.5))
    for index, (algorithm, color) in enumerate(zip(ALGORITHMS, COLORS)):
        values = [float(summary_by_algorithm[algorithm][scenario][metric]) for scenario in SCENARIOS]
        axis.bar(x + (index - 1) * width, values, width, label=algorithm, color=color)
    axis.set_title(title)
    axis.set_ylabel(ylabel)
    axis.set_xticks(x)
    axis.set_xticklabels(SCENARIOS)
    axis.legend()
    axis.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_metric("deadline_misses", "Deadline Misses", "count")

In [ ]:
plot_metric("packet_losses", "Packet Losses", "count")

In [ ]:
plot_metric("avg_latency", "Average Latency", "seconds")

In [ ]:
plot_metric("avg_energy", "Average Total System Energy", "joules")

`avg_energy` is vehicle energy plus Fog or Cloud infrastructure energy. The detailed CSV files preserve both components separately.